# SnowIn core workflow

This self-contained notebook introduces the main xarray-native SnowIn APIs without requiring a downloaded NISAR product. It demonstrates phase-to-dSWE conversion, reference phase, corrections, support layers, temporal accumulation, metrics, and basic plotting.

In [1]:
from html import escape
import math

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
import xarray as xr

import snowin
from snowin import (
    accumulate_dswe,
    build_support_dataset,
    compose_support_mask,
    compute_dswe,
    compute_metrics,
    reference_phase,
    summarize_support,
)
from snowin.corrections import PhaseCorrectionConfig, apply_phase_corrections
from snowin.reference import apply_reference_phase

def _format_metadata_value(value):
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.6g}"
    return str(value)


def _display_metadata(title, attrs, fields):
    rows = "".join(
        f"<tr><th><code>{escape(field)}</code></th><td>{escape(_format_metadata_value(attrs.get(field, 'not present')))}</td></tr>"
        for field in fields
    )
    display(HTML(
        f"<h3>{escape(title)}</h3><table><thead><tr><th>Field</th><th>Value</th></tr></thead><tbody>{rows}</tbody></table>"
    ))


display(HTML(f"<h2>SnowIn {escape(snowin.__version__)}</h2><p>Core xarray workflow example.</p>"))

SnowIn 0.1.0


## 1. Create a normalized phase pair

SnowIn's canonical phase is `secondary_minus_reference`, and angles are stored in radians.

In [2]:
rng = np.random.default_rng(7)
ny, nx = 60, 80
y = np.arange(ny, dtype=float)
x = np.arange(nx, dtype=float)
coords = {"y": y, "x": x}

incidence_values = np.broadcast_to(
    np.deg2rad(32.0 + 5.0 * np.sin(x[None, :] / 18.0)), (ny, nx)
)
incidence = xr.DataArray(
    incidence_values,
    dims=("y", "x"),
    coords=coords,
    name="incidence_angle",
    attrs={"units": "rad", "incidence_angle_reference": "local"},
)
phase_values = 0.8 * np.sin(x[None, :] / 14.0) + 0.3 * np.cos(y[:, None] / 11.0)
phase_values += 0.04 * rng.normal(size=(ny, nx))
phase_values[:4, :6] = np.nan
phase = xr.DataArray(
    phase_values,
    dims=("y", "x"),
    coords=coords,
    name="phase",
    attrs={"units": "rad", "phase_difference_definition": "secondary_minus_reference"},
)
pair = xr.Dataset(
    {"phase": phase, "incidence_angle": incidence},
    attrs={
        "phase_difference_definition": "secondary_minus_reference",
        "reference_time": "2025-01-01T00:00:00Z",
        "secondary_time": "2025-01-13T00:00:00Z",
    },
)
pair

<xarray.Dataset> Size: 78kB
Dimensions:          (y: 60, x: 80)
Coordinates:
  * y                (y) float64 480B 0.0 1.0 2.0 3.0 ... 56.0 57.0 58.0 59.0
  * x                (x) float64 640B 0.0 1.0 2.0 3.0 ... 76.0 77.0 78.0 79.0
Data variables:
    phase            (y, x) float64 38kB nan nan nan ... -0.3271 -0.3773 -0.2403
    incidence_angle  (y, x) float64 38kB 0.5585 0.5634 0.5682 ... 0.4774 0.4758
Attributes:
    phase_difference_definition:  secondary_minus_reference
    reference_time:               2025-01-01T00:00:00Z
    secondary_time:               2025-01-13T00:00:00Z

## 2. Compute pairwise dSWE

The canonical kernel returns dSWE in metres and preserves the grid and metadata.

In [3]:
wavelength_m = 0.238403545
dswe = compute_dswe(pair["phase"], pair["incidence_angle"], wavelength_m=wavelength_m)
display(dswe)
assert dswe.attrs["units"] == "m"
assert dswe.dims == pair["phase"].dims

<xarray.DataArray 'dswe' (y: 60, x: 80)> Size: 38kB
array([[        nan,         nan,         nan, ..., -0.00605617,
        -0.00504909, -0.00254295],
       [        nan,         nan,         nan, ..., -0.00646334,
        -0.00342212, -0.00447754],
       [        nan,         nan,         nan, ..., -0.00592128,
        -0.00495134, -0.00523486],
       ...,
       [ 0.00345561,  0.00538441,  0.00494321, ..., -0.00936166,
        -0.00779165, -0.00787465],
       [ 0.00216748,  0.00335666,  0.00541374, ..., -0.0090124 ,
        -0.0090227 , -0.00740501],
       [ 0.00422143,  0.00428298,  0.00550009, ..., -0.00709658,
        -0.00819229, -0.00522087]], shape=(60, 80))
Coordinates:
  * y        (y) float64 480B 0.0 1.0 2.0 3.0 4.0 ... 55.0 56.0 57.0 58.0 59.0
  * x        (x) float64 640B 0.0 1.0 2.0 3.0 4.0 ... 75.0 76.0 77.0 78.0 79.0
Attributes:
    units:                        m
    phase_difference_definition:  secondary_minus_reference
    quantity:                     pairwise_dSWE
    long_name:                    pairwise change in snow water equivalent
    incidence_angle_reference:    local
    wavelength_m:                 0.238403545
    alpha:                        1.0
    snow_path_constant:           1.59
    equation:                     dSWE = phase * wavelength_m / (2*pi*alpha*(...
    scientific_reference:         Leinss et al. (2015), Eq. 18

## 3. Apply explicit reference phase

Reference inputs are supplied by the caller. SnowIn does not choose stations or dates.

In [4]:
station_ids = xr.DataArray(
    ["station_a", "station_b", "station_c", "station_d"], dims="station"
)
observed = xr.DataArray(
    [0.37, 0.41, np.nan, 0.36], dims="station", attrs={"units": "rad"}
)
expected = xr.DataArray(
    [0.10, 0.12, 0.20, 0.11], dims="station", attrs={"units": "rad"}
)
weights = xr.DataArray([0.8, 0.6, 0.4, 0.9], dims="station", attrs={"units": "1"})
exclusion_reason = xr.DataArray(
    ["eligible", "eligible", "missing_station_phase", "eligible"], dims="station"
)

referenced = reference_phase(
    pair,
    observed,
    expected,
    weights,
    contributor_id=station_ids,
    exclusion_reason=exclusion_reason,
)
_display_metadata(
    "Reference-phase summary",
    referenced.attrs,
    [
        "phase_reference_status",
        "phase_reference_offset_rad",
        "phase_reference_eligible_count",
    ],
)
referenced_dswe = compute_dswe(
    referenced["phase_referenced"],
    referenced["incidence_angle"],
    wavelength_m=wavelength_m,
)
assert referenced.attrs["phase_reference_status"] == "SUPPORTED"

reference offset [rad]: 0.2673913043478261
eligible contributors: 3


The older NumPy compatibility helper is still available for simple diagnostics, but it is not an implicit default in the xarray workflow.

In [5]:
legacy_reference = apply_reference_phase(pair["phase"].values, strategy="median_valid")
legacy_reference.diagnostics

{'strategy': 'median_valid',
 'reference_value_rad': 0.04632545113275424,
 'reference_pixels': 4776,
 'reference_fraction': 0.995}

## 4. Apply an explicitly signed phase correction

Corrections require a declared sign and remain separate from reference phase.

In [6]:
correction_layer = np.full((ny, nx), 0.02)
correction = apply_phase_corrections(
    pair["phase"].values,
    wet_tropo=correction_layer,
    config=PhaseCorrectionConfig(apply_wet_tropo=True, wet_tropo_sign="subtract"),
)
_display_metadata(
    "Phase-correction diagnostics",
    correction.diagnostics,
    list(correction.diagnostics),
)
assert np.allclose(
    correction.phase_corrected[10:, 10:],
    pair["phase"].values[10:, 10:] - 0.02,
    equal_nan=True,
)

{'config': {'apply_ionosphere': False, 'ionosphere_sign': None, 'apply_wet_tropo': True, 'wet_tropo_sign': 'subtract', 'apply_hydro_tropo': False, 'hydro_tropo_sign': None, 'remove_planar_ramp': False}, 'phase_raw_finite_fraction': 0.995, 'phase_corrected_finite_fraction': 0.995, 'applied_terms': ('wet_tropo',), 'wet_tropo_stats': {'valid_n': 4800, 'mean': 0.02, 'std': 0.0, 'min': 0.02, 'max': 0.02}, 'phase_corrected_stats': {'valid_n': 4776, 'mean': -0.038074890432627836, 'std': 0.6205900189281497, 'min': -1.2101691375965182, 'max': 1.1664752433550563}}


## 5. Build and summarize named support layers

In [7]:
product_valid = np.isfinite(pair["phase"]).rename("product_valid")
geometry_valid = (
    (pair["incidence_angle"] >= 0) & (pair["incidence_angle"] < math.pi / 2)
).rename("geometry_valid")
reference_supported = xr.ones_like(product_valid, dtype=bool).rename(
    "reference_supported"
)
pairwise_supported = np.isfinite(referenced_dswe).rename("pairwise_supported")
support = build_support_dataset(
    product_valid=product_valid,
    geometry_valid=geometry_valid,
    reference_supported=reference_supported,
    pairwise_supported=pairwise_supported,
)
support_mask = compose_support_mask(
    support,
    ["product_valid", "geometry_valid", "reference_supported", "pairwise_supported"],
)
summarize_support(support)

<xarray.Dataset> Size: 464B
Dimensions:           (support_category: 4)
Coordinates:
  * support_category  (support_category) <U19 304B 'product_valid' ... 'pairw...
Data variables:
    total_count       (support_category) int64 32B 4800 4800 4800 4800
    known_count       (support_category) int64 32B 4800 4800 4800 4800
    supported_count   (support_category) int64 32B 4776 4800 4800 4776
    support_fraction  (support_category) float64 32B 0.995 1.0 1.0 0.995
    known_fraction    (support_category) float64 32B 1.0 1.0 1.0 1.0
Attributes:
    support_semantics:             named support summary
    support_fraction_definition:   supported_count / known_count
    known_fraction_definition:     known_count / total_count
    unknown_support_is_not_false:  True

## 6. Accumulate a chronological path

In [ ]:
def temporal_edge(values, reference_time, secondary_time):
    values = values.rename("dswe").copy()
    values.attrs = {
        **values.attrs,
        "units": "m",
        "quantity": "pairwise_dSWE",
        "phase_difference_definition": "secondary_minus_reference",
    }
    return xr.Dataset(
        {"dswe": values},
        attrs={
            "temporal_edge": "reference_to_secondary",
            "reference_time": reference_time,
            "secondary_time": secondary_time,
        },
    )


edge_1 = temporal_edge(referenced_dswe, "2025-01-01T00:00:00Z", "2025-01-13T00:00:00Z")
edge_2 = temporal_edge(
    referenced_dswe * 0.5, "2025-01-13T00:00:00Z", "2025-01-25T00:00:00Z"
)
path = accumulate_dswe([edge_1, edge_2])
path

## 7. Compute metrics and make quick plots

In [ ]:
observed_dswe = referenced_dswe.rename("observed_dswe")
estimated_dswe = (referenced_dswe + 0.005).rename("estimated_dswe")
observed_dswe.attrs = {"units": "m"}
estimated_dswe.attrs = {"units": "m"}
metrics = compute_metrics(observed_dswe, estimated_dswe, support=support_mask)
metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
pair["phase"].plot(ax=axes[0], cmap="viridis")
axes[0].set_title("Canonical phase [rad]")
referenced_dswe.plot(ax=axes[1], cmap="RdBu_r")
axes[1].set_title("Referenced dSWE [m]")
support_mask.plot(ax=axes[2], cmap="Greens")
axes[2].set_title("Explicit support")
plt.show()

assert path.attrs["temporal_edge_count"] == 2
assert (
    metrics.attrs["support_policy"]
    == "finite_observed_and_estimated_and_explicit_support"
)
display(HTML("<h3 style='color: #18794e;'>Core SnowIn workflow checks passed.</h3>"))